# Hidden Markov Model
Hidden Markov Models (HMMs) contain hidden states that we are trying to infer from observed data. This is useful in bioinformatics, because the observed data is what we can directly measure, like sequenced DNA. On the other hand, the hidden states represent the underlying biological context we are trying to uncover or infer. We will use the Viterbi algorithm to find the most likely sequence of hidden states given a sequence of observations.

In [1]:
import numpy as np
from hmm_utils import HiddenMarkovModel

In [2]:
class ForwardBackward(HiddenMarkovModel):
    def forward_matrix(self, observations):
        """
        The algorithm computes the forward probability matrix. Each entry represents the probability of having emitted the observed symbols up to the particular position and state.

        :param observations: list of observed sequences
        :return:
        prob_matrix: np.ndarray - Forward probability matrix of shape (num_states x len(observations)) in log space
        """
        prob_matrix = np.zeros((len(self.states), len(observations)), dtype = float)

        ####### Iteration ########
        for i, observation in enumerate(observations):
            for j, state in enumerate(self.states):
                # Get initial and emission probabilities for current state
                state_init_probs = self.initial_probs[state]
                state_emit_probs = self.get_emission_probs(state)

                # calculate probabilities
                # Remember log(x*y) = log(x) + log(y)
                # first column
                if i == 0:
                    state_prob = np.log(state_init_probs) + np.log(state_emit_probs[observation])
                    prob_matrix[j][i] = state_prob

                else:
                    joint_probs = [prob_matrix[k][i-1] +
                                   np.log(self.get_transition_probs(prev_state)[state]) +
                                   np.log(state_emit_probs[observation])
                                   for k, prev_state in enumerate(self.states)]

                    sum_prob = np.logaddexp.reduce(joint_probs)
                    prob_matrix[j][i] = sum_prob

        return prob_matrix

    def backward_matrix(self, observations):
        """
        The algorithm computes the backward probability matrix. Each entry represents the probability of having emitted the observed symbols from the current positon to the end of the sequence given a specific state.

        :param observations: list of observed sequences
        :return:
        prob_matrix: np.ndarray - Forward probability matrix of shape (num_states x len(observations)) in log space
        """
        prob_matrix = np.zeros((len(self.states), len(observations)), dtype = float)

        for i in range(len(observations)-2, -1, -1):
            for j, state in enumerate(self.states):
                joint_probs = [prob_matrix[k][i+1] +
                           np.log(self.get_transition_probs(state)[next_state]) +
                           np.log(self.get_emission_probs(next_state)[observations[i+1]])
                           for k, next_state in enumerate(self.states)]

                prob_matrix[j][i] = np.logaddexp.reduce(joint_probs)

        return prob_matrix

    def sequence_probability(self, prob_matrix):
        """
        Computes the total probability of the observation sequence from either the forward probability matrix.

        :param prob_matrix:  np.ndarray - Forward probability matrix of shape (num_states x len(observations))
        :return:
            float - Log probability of the observation sequence
        """
        return np.logaddexp.reduce(prob_matrix[:, -1])


    def forward_backward(self, observations):
        """
        The algorithm computes the forward-backward posterior matrix by combining the forward and backward matrices, normalized by the total probability of the observation sequence.

        :param observations: list of observed sequences
        :return:
        posterior_matrix: np.ndarray - posterior probability matrix of shape (num_states x len(observations)) in log space
        """
        self.posterior_matrix = np.zeros((len(self.states), len(observations)), dtype = float)

        forward_matrix = self.forward_matrix(observations)
        backward_matrix = self.backward_matrix(observations)

        final_col_prob = self.sequence_probability(forward_matrix)

        for i in range(len(observations)):
            for j in range(len(self.states)):
                self.posterior_matrix[j][i] = forward_matrix[j][i] + backward_matrix[j][i] - final_col_prob

        return self.posterior_matrix

    def posterior_decode(self, observations):
        """
        Decodes the most probable state at each position independently using posterior decoding
        :param observations: list of observed sequences
        :return:
        path: list - The most probable state at each position
        """
        self.forward_backward(observations)
        path = []
        for i in range(len(observations)):
            best_state = np.argmax(self.posterior_matrix[:, i])
            path.append(self.states[best_state])

        return path

    def get_posterior_at_index(self, position):
        """
        Returns probabilities of a particular position (index) for all hidden states
        :param position: int index of desired position to check probabilities
        :return:
        list of probabilties
        
        """
        return self.posterior_matrix[:,position]


In [3]:
# Example observation sequence following the powerpoint
obs = "ACGCGATC"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.1,
    "G": 0.9
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.6, "G": 0.4},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.4, "C": 0.1, "G": 0.1, "T": 0.4}
}

hmm = ForwardBackward(init_probs, trans_probs, emit_probs)

print(f"Forward: {hmm.forward_matrix(obs)}\n\nBackward: {hmm.backward_matrix(obs)}")

print(f"\nPosterior Matrix: ")
fb_matrix = hmm.forward_backward(obs)
print(fb_matrix)

print(f"\nPosterior Decoded: {hmm.posterior_decode(obs)}")
viterbi = hmm.viterbi_algorithm(obs)
print(f"\nViterbi Matrix: {viterbi}")

index_to_check = 4
print(f"\nProbabilities of states at index: {index_to_check}")
posteriors_at_index = hmm.get_posterior_at_index(index_to_check)
for state, prob in zip(hmm.states, posteriors_at_index):
    print(f"{state} : {prob}")



Forward: [[ -4.60517019  -4.08637639  -5.23178084  -6.55181661  -7.917803
  -10.68397931 -12.96631674 -13.07609226]
 [ -1.02165125  -3.41732676  -5.62017689  -7.52408958  -9.15496622
   -9.24781431 -10.16898167 -12.55018913]]

Backward: [[ -9.82510559  -8.4447688   -7.07732394  -5.75233427  -4.58615218
   -3.28075123  -1.27296568   0.        ]
 [-11.16497558  -9.69183725  -8.07448111  -6.19061817  -4.00457699
   -3.00376445  -2.04022083   0.        ]]

Posterior Matrix: 
[[-2.34446282 -0.44533223 -0.22329182 -0.21833791 -0.41814222 -1.87891758
  -2.15346945 -0.9902793 ]
 [-0.10081387 -1.02335106 -1.60884504 -1.62889479 -1.07373024 -0.1657658
  -0.12338953 -0.46437617]]

Posterior Decoded: ['G', 'I', 'I', 'I', 'I', 'G', 'G', 'G']
['I', 'G']

Viterbi Matrix: [['G', 'I', 'I', 'I', 'I', 'G', 'G', 'G']]

Probabilities of states at index: 4
I : -0.41814221787486083
G : -1.0737302429154543
